# **Extraer la Data del CSV e Instalación de Librerías**

Fragmento para declarar e instalar la librerías que se implementaran en el notebook.

In [1]:
import pandas as pd

Importar la data del archivo CSV, indicando la ruta del archivo, y mostrar parcialmente su contenido.

In [2]:
data = pd.read_csv(filepath_or_buffer='./data/backloggd_indie_games.csv', encoding="UTF-8")
data

,Title,Developer,Release_Year,Avg_Rating,Plays,Rating_Count,Genres,Platforms,Avg_Time
0,Hollow Knight,Team Cherry,2017.0,4.4,181K,124K,"Adventure, Indie, Platform","Windows PC, Mac, Linux, Nintendo Switch 2, Pla...",59h
1,Undertale,tobyfox,2015.0,4.3,158K,106K,"Adventure, Indie, Puzzle, RPG","Windows PC, Mac, Linux, Nintendo Switch, PlayS...",16h
2,Hades,Supergiant Games,2020.0,4.2,156K,104K,"Adventure, Brawler, Indie, RPG","Windows PC, Mac, PlayStation 5, Xbox Series X|...",48h
3,Stardew Valley,ConcernedApe,2016.0,4.2,152K,93K,"Adventure, Indie, RPG, Simulator, Strategy","Windows PC, Mac, Linux, Nintendo Switch 2, Nin...",129h
4,Among Us,Innersloth,2018.0,3.1,149K,82K,"Indie, Strategy","Windows PC, PlayStation 5, Xbox Series X|S, Ni...",27h
...,...,...,...,...,...,...,...,...,...
48765,Cheated,Tbjbu2,2026.0,NaN,0.0,0.0,"Adventure, Indie, Simulator, Strategy",Windows PC,- h
48766,Outer Space Bar,OZNA STUDIO,2026.0,NaN,0.0,0.0,"Arcade, Indie",Windows PC,- h
48767,Astro Amigo,NaN,2026.0,NaN,0.0,0.0,"Adventure, Indie","Windows PC, Mac",- h
48768,PNR: Point of No return,NaN,2026.0,NaN,0.0,0.0,Indie,Windows PC,- h


# **Limpieza de Datos**

Antes de interactuar con la data, se identificado valores nulos en las siguientes columnas:

- *Developer*: Desarrollador
- *Release_Year*: Año de Lanzamiento
- *Avg_Rating*: Promedio de la Valoración
- *Platforms*: Plataformas

In [3]:
data = data.dropna(subset=[
    'Developer',
    'Release_Year',
    'Avg_Rating',
    'Platforms'
])

Basado en el contenido del dataset, es necesario hacer transformaciones en los datos de las siguientes columnas del dataset:

- **Avg_Time**: El valor de las horas promedio para finalizar el juego, esta acompañado en una 'h' para indicar que el tiempo esta en horas. Para ello, se elimina ese caracter para trabajar esta columna en datos numericos.

- **Rating_Count y Plays**: El valor de estas columnas esta acompañado de una 'K' para indicar que el valor esta simplificado y se representa en miles. Entonces, se elimina ese caracter, cambiamos el tipo de dato a numerico y multiplicamos por 1000 para mantener el valor original.

In [4]:
def transfrom_val(value):

    if value[-1:] == 'K':
        value_str = value.replace('K', '')
        return float(value_str) * 1000

    return float(value)

In [5]:
for i, row in data.iterrows():

    rate_count = transfrom_val(row['Rating_Count'])
    plays = transfrom_val(row['Plays'])

    time = row['Avg_Time'].replace('h', '').replace('- ', '0')

    data.loc[i, 'Rating_Count'] = rate_count
    data.loc[i, 'Plays'] = plays
    data.loc[i, 'Avg_Time'] = float(time)

Se elimina aquellas filas que cumplen con las siguientes condiciones (las que se expresa en el siguiente fragmento de código):

- La cantidad de veces que se inicio sesion en un juego sea igual a 0.
- El contador de calificaciones sea igual a 0.
- El tiempo de promedio de horas ocupadas para terminar un juego sea 0 o no hay datos reportados.

In [6]:

data = data[data['Plays'] > 0]
data = data[data['Rating_Count'] > 0]
data = data[data['Avg_Time'] > 0]

data

,Title,Developer,Release_Year,Avg_Rating,Plays,Rating_Count,Genres,Platforms,Avg_Time
0,Hollow Knight,Team Cherry,2017.0,4.4,181000.0,124000.0,"Adventure, Indie, Platform","Windows PC, Mac, Linux, Nintendo Switch 2, Pla...",59.0
1,Undertale,tobyfox,2015.0,4.3,158000.0,106000.0,"Adventure, Indie, Puzzle, RPG","Windows PC, Mac, Linux, Nintendo Switch, PlayS...",16.0
2,Hades,Supergiant Games,2020.0,4.2,156000.0,104000.0,"Adventure, Brawler, Indie, RPG","Windows PC, Mac, PlayStation 5, Xbox Series X|...",48.0
3,Stardew Valley,ConcernedApe,2016.0,4.2,152000.0,93000.0,"Adventure, Indie, RPG, Simulator, Strategy","Windows PC, Mac, Linux, Nintendo Switch 2, Nin...",129.0
4,Among Us,Innersloth,2018.0,3.1,149000.0,82000.0,"Indie, Strategy","Windows PC, PlayStation 5, Xbox Series X|S, Ni...",27.0
...,...,...,...,...,...,...,...,...,...
18882,Dino Path Trail,Void Pointer,2025.0,1.9,7.0,5.0,Indie,Windows PC,1.0
19587,After You,k148 Game Studio,2020.0,2.3,6.0,6.0,"Adventure, Indie, Puzzle","Windows PC, PlayStation 5, Xbox Series X|S, Ni...",1.0
20025,Kernelbay,Nexent,2026.0,2.4,6.0,5.0,Indie,Windows PC,15.0
20068,Typical,TsunamiGames,2018.0,1.9,6.0,5.0,"Adventure, Indie, Visual Novel","Windows PC, Mac, Linux",16.0


#   Análisis de Datos

## El juego del Año (Categoria: Indie)

El siguente fragmento se obtiene el mejor juego de indie de cada año registrado en el dataset.
La ejecución se basa en los siguientes pasos:

- Se filtra los juegos por el año del lanzamiento 
- Se ordena, de manera descendente, el promedio de la valoración
- Se obtiene el primer registro

In [7]:
# Obtengo el valor minimo y maximo del año de lanzamiento
sup_years = int(data['Release_Year'].max())
inf_years = int(data['Release_Year'].min())

# Creo un dataframe 'games' para guardar los juegos del año
games = pd.DataFrame(columns=['Game', 'Year', 'Rating'])

# Itero cada año desde el valor minimo hasta el maximo
for i in range(inf_years, sup_years):

    # Filtro por el año de lanzamiento 
    datos = data[data['Release_Year'] == i]

    # Aseguro que los resultados no esten vacios
    if not datos.empty:

        # Se ordena por valoracion, obtengo el primer registro y lo guardo en 'games'
        datos = datos.sort_values(by='Avg_Rating', ascending=False)
        datos = datos.iloc[0]
        games.loc[i - inf_years] = {'Game': datos['Title'], 'Year': i, 'Rating': datos['Avg_Rating']}

games

,Game,Year,Rating
0,Alter Ego,1986,3.4
1,Shadowgate,1987,3.3
2,Wasteland,1988,2.9
5,Supaplex,1991,3.6
6,Star Control II,1992,4.3
8,Umihara Kawase,1994,3.6
9,Pulstar,1995,3.2
10,Snood,1996,3.1
11,Game Dev Story,1997,3.5
12,LSD: Dream Emulator,1998,3.8


## Mejor Juego con Filtro a la Plataforma

Todos los juegos pueden ser jugados en al menos 1 o más plataformas disponibles. Para obtener los mejores juegos de una plataforma, se indica una plataforma como filtro y se ordenara, de forma descendente, el promedio de la valoración.

Esta el caso de que esta la marca de una plataforma. Si uno de las plataformas que tiene un juego contiene la palabra *PlayStation*, entonces deberia mostrar los siguientes resultados a coincidir:

- *PlayStation 1*
- *PlayStation 2*
- *PlayStation 3*
- *PlayStation 4*
- *PlayStation 5*

In [8]:
brand = 'PlayStation'


games = pd.DataFrame(columns=['Game', 'Year', 'Rating', 'Platform'])

size = 0

for i, row in data.iterrows():
    group_plat = row['Platforms'].split(',')

    for plat in group_plat:

        plat = plat.lstrip()

        if plat.startswith(brand):

            games.loc[size] = {'Game': row['Title'], 'Year': row['Release_Year'], 'Rating': row['Avg_Rating'], 'Platform': row['Platforms']}
            size += 1
            break

games = games.sort_values(by='Rating', ascending=False)
games

,Game,Year,Rating,Platform
30,Disco Elysium: The Final Cut,2021.0,4.6,"Windows PC, Mac, PlayStation 5, Xbox Series X|..."
9,Clair Obscur: Expedition 33,2025.0,4.6,"Windows PC, PlayStation 5, Xbox Series X|S"
22,Outer Wilds,2019.0,4.6,"Windows PC, PlayStation 5, Xbox Series X|S, Ni..."
61,Deltarune,2025.0,4.5,"Windows PC, Mac, Nintendo Switch 2, PlayStatio..."
882,The Binding of Isaac: Repentance,2021.0,4.5,"Windows PC, PlayStation 5, Xbox Series X|S, Ni..."
...,...,...,...,...
3290,Color Slayer,2019.0,1.2,"Windows PC, PlayStation 4, PlayStation Vita"
3178,Grey Skies: A War of the Worlds Story,2020.0,1.1,"Windows PC, Nintendo Switch, PlayStation 4, Xb..."
3040,Energy Invasion,2018.0,1.0,"Windows PC, Mac, Linux, Nintendo Switch, PlayS..."
2562,Albedo: Eyes from Outer Space,2015.0,1.0,"Windows PC, Mac, PlayStation 4, Xbox One"


## Los mejores Géneros Secundarios

Hallar el ranking de los géneros secundarios dentro de los juegos indies del dataset. Dentro de ese ranking, se recolecta la siguiente información:

- Nombre del género secundario
- Frecuecia o cantidad de veces que un genero es mencionado
- Porcentaje que representa un género dentro del dataset 

In [20]:
generos = data['Genres'].str.split(', ')
generos = generos.explode().unique()
generos = generos[generos != 'Indie']

genre_stat = pd.DataFrame(columns=['Generos', 'Contador', 'Porcentaje'])
size = 0
total = len(data)

for genero in generos:

    contador = int(data['Genres'].str.contains(genero, na=False).sum())

    percent = round((contador / total) * 100, 2)

    genre_stat.loc[size] = {'Generos': genero, 'Contador': contador, 'Porcentaje': percent}

    size += 1    

genre_stat = genre_stat.sort_values(by='Porcentaje', ascending=False)
genre_stat

,Generos,Contador,Porcentaje
0,Adventure,5213,57.91
2,Puzzle,2277,25.29
5,Simulator,2199,24.43
3,RPG,2198,24.42
6,Strategy,2009,22.32
1,Platform,1517,16.85
7,Arcade,1285,14.27
8,Shooter,1264,14.04
14,Point-and-Click,593,6.59
13,Visual Novel,564,6.27
